# Find the attention sink in ANY VLM, remove it, then compute the distribution

Every importance map we have produced peaked on the **same token** regardless of image or
question — patch 80 on SmolVLM2. That token is an *attention sink*: a slot the decoder uses to
park surplus attention. It carries no image content, but it eats most of the probability budget.

## How the detector works

A mean-based rule cannot find it. A sink and a genuinely important patch can both have high
*average* incoming attention. The difference is the **floor**:

* a **content** patch is attended to strongly by a **few** queries and ignored by the rest
* a **sink** is attended to by **every** query — it has a high floor

So score each image token by the low **quantile** of the attention it receives across text query
rows, not the mean:

```
score[j] = mean over (layer, head) of  quantile_0.1 over text queries ( A[query, j] )
```

That needs **one forward pass** and nothing model-specific — it reads the attention tensor any VLM
already produces.

**One trap, which the first version of this code fell into.** Chat templates put text on BOTH sides
of the image block (`<|im_start|>User:` before it, the question after). Under causal masking those
leading rows have *exactly zero* attention to every image token. With ~15% such rows a 0.1 quantile
reads straight out of the blind zone and returns 0 for every patch — an all-zero score vector, no
sink found. `sink_scores` now drops rows whose attention mass over image columns is <= `min_mass`
before taking the quantile. The sanity print below exists to catch exactly this.

## Then remove it *before* the softmax

`image_importance(..., cand_mask=...)` masks sinks to `-inf` **before** the softmax over image
tokens. Subtracting a sink afterwards cannot undo the mass it already stole from every other patch.

## 1. Setup

In [ ]:
# optional -- only needed to probe with your own WEAR-VQA images.
# Without it the notebook falls back to public COCO images, which is fine:
# the sink is a property of the MODEL, not of your data.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive not mounted:", e)

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words matplotlib
!rm -rf /content/text_vision_attention_map
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git /content/text_vision_attention_map
%cd /content/text_vision_attention_map

In [ ]:
import importlib.util, os, sys, math, glob, json
import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, os.getcwd())

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S = _load("probe_smolvlm", "tests/probe_smolvlm.py")
import rater_selection as RS
import visual_selection as VS

for fn in ("sink_scores", "detect_sinks", "aggregate_sink_scores", "sink_report"):
    assert hasattr(VS, fn), f"clone is stale - visual_selection has no {fn}"

N_PROBE  = 6          # examples used to estimate the sink (it is a MODEL property)
DATA_ROOT = "/content/drive/MyDrive/wearvqa_gaze_only"
print("ready")

In [ ]:
# Probe examples: your WEAR-VQA data if Drive is mounted, otherwise public images.
# Deliberately DIFFERENT scenes and questions -- a sink is whatever survives that.
samples = []
if os.path.isdir(DATA_ROOT):
    for jp in sorted(glob.glob(os.path.join(DATA_ROOT, "*", "*.json")))[:N_PROBE * 3:3]:
        meta = json.load(open(jp))
        img = jp[:-5] + ".jpg"
        if os.path.exists(img):
            samples.append((img, meta["question"]))
        if len(samples) == N_PROBE:
            break

if not samples:
    print("Drive data not found - falling back to public images")
    base = "http://images.cocodataset.org/val2017/"
    samples = [
        (base + "000000039769.jpg", "How many cats are in the image?"),
        (base + "000000000285.jpg", "What animal is this?"),
        (base + "000000000724.jpg", "What does the sign say?"),
        (base + "000000001000.jpg", "What is the person holding?"),
        (base + "000000001268.jpg", "What colour is the wall?"),
        (base + "000000001503.jpg", "What is on the table?"),
    ][:N_PROBE]

print(f"{len(samples)} probe examples")
for p, q in samples:
    print(f"   {os.path.basename(p):<20} {q}")

## 2. Detect the sink

One forward pass per example. `ProbeOutput` already carries the full attention plus the
image/text masks, so this section is model-agnostic — swap the probe and it still runs.

In [ ]:
scores, mean_scores = [], []
for path, q in samples:
    o = S.make_smolvlm_output(image=S.load_image(path), question=q)
    assert o is not None, "probe failed"
    # the detector: low-quantile floor of incoming attention
    scores.append(VS.sink_scores(o.post_softmax, o.image_token_mask, o.text_token_mask,
                                 is_post_softmax=True))
    # naive comparison: plain MEAN incoming attention
    tp = torch.nonzero(o.text_token_mask).squeeze(-1)
    vp = torch.nonzero(o.image_token_mask).squeeze(-1)
    acc = torch.zeros(vp.numel())
    for l, A in o.post_softmax.items():
        acc += A.float()[:, tp][:, :, vp].mean(dim=1).mean(dim=0)
    mean_scores.append(acc / len(o.post_softmax))
    del o

score      = VS.aggregate_sink_scores(scores)
mean_score = VS.aggregate_sink_scores(mean_scores)
L_v = score.numel()
G   = int(round(math.sqrt(L_v)))

sinks = VS.detect_sinks(score)                 # robust median+MAD rule
print(VS.sink_report(score, sinks))
print()
print(f"quantile-floor detector picks : {torch.nonzero(sinks).squeeze(-1).tolist()}")
print(f"quantile top-3                : {torch.topk(score, 3).indices.tolist()}")
print(f"naive MEAN top-3              : {torch.topk(mean_score, 3).indices.tolist()}")

# guard against the blind-row bug: a median of exactly 0 means the query rows that
# cannot see the image were not filtered and the quantile read from the blind zone
print(f"\nsanity: median sink score = {float(score.median()):.2e}  (must be > 0)")
assert float(score.median()) > 0, (
    "all-zero score vector -- blind query rows poisoned the quantile")

In [ ]:
# per-example stability: a real sink is the same token every time
print("argmax of the sink score, per probe example:")
for (p, q), s in zip(samples, scores):
    print(f"   {os.path.basename(p):<20} -> token {int(s.argmax())}")

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for a, (t, v) in zip(ax, [("sink score (quantile floor)", score),
                          ("naive mean attention", mean_score),
                          ("detected sinks", sinks.float())]):
    im = a.imshow(v.reshape(G, G), cmap="magma")
    a.set_title(t, fontsize=10); a.set_xticks([]); a.set_yticks([])
    fig.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout(); plt.show()

## 3. Remove it, then compute the distribution

`cand_mask` is the complement of the sink mask. It is applied **inside** `image_importance`,
before the softmax over image tokens.

In [ ]:
path, q = samples[0]
o = S.make_smolvlm_output(image=S.load_image(path), question=q)
maps, tpos, _ = RS.sliced_maps_from_full(o.raw_scores, o.image_token_mask, o.text_token_mask)
tt = S._load_smolvlm("HuggingFaceTB/SmolVLM2-2.2B-Instruct")[1].tokenizer.convert_ids_to_tokens(
    o.input_ids[tpos].tolist())
rr = RS.select_important_text_tokens(maps, text_tokens=tt, tokenizer=None, question=q, pct=0.5)

cand = VS.candidate_mask(L_v, exclude=[sinks])
before, *_ = VS.image_importance(maps, rr.rater_mask)
after,  *_ = VS.image_importance(maps, rr.rater_mask, cand_mask=cand)

def summarise(name, p):
    nz = p[p > 0]
    ent = float(-(nz * nz.log()).sum())
    print(f"{name:<26} argmax {int(p.argmax()):>3}   max {float(p.max()):.4f}   "
          f"entropy {ent:.3f}   mass on sinks {float(p[sinks].sum()):.4f}")

print(f"question: {q}\nraters  : {rr.kept_tokens(tt)}\n")
summarise("WITH sink (before)", before)
summarise("sink excluded (after)", after)
print(f"\nprobability freed up and redistributed: {float(before[sinks].sum()):.1%}")
print(f"largest real patch gained: "
      f"{float(after[cand].max() - before[cand].max()):+.4f}")

In [ ]:
img = S.load_image(path)
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, (t, v) in zip(ax, [("image", None),
                          ("importance WITH sink", before),
                          ("importance, sink removed", after)]):
    a.imshow(img); a.axis("off"); a.set_title(t, fontsize=10)
    if v is not None:
        from PIL import Image as _I
        h = v.reshape(G, G).numpy()
        h = (h - h.min()) / (np.ptp(h) + 1e-9)
        a.imshow(np.array(_I.fromarray((h * 255).astype("uint8")).resize(img.size)),
                 cmap="jet", alpha=0.5)
plt.tight_layout(); plt.show()

## 4. Freeze the mask and reuse it

The sink is a property of the **model**, not of your dataset, so estimate it once and freeze it.
Reusing a frozen mask also keeps labels independent of which examples happened to be in a batch.

In [ ]:
OUT = "/content/sink_mask_smolvlm2.pt"
torch.save({"model": "HuggingFaceTB/SmolVLM2-2.2B-Instruct", "L_v": L_v,
            "sink_mask": sinks, "sink_score": score, "n_probe": len(samples)}, OUT)
print("saved", OUT)
if os.path.isdir("/content/drive/MyDrive"):
    import shutil; shutil.copy(OUT, "/content/drive/MyDrive/"); print("copied to Drive")

print("\nuse it as:")
print("   b = torch.load('sink_mask_smolvlm2.pt')")
print("   cand = VS.candidate_mask(b['L_v'], exclude=[b['sink_mask'], fovea_mask])")
print("   imp, *_ = VS.image_importance(maps, rater_mask, cand_mask=cand)")

## 5. A different VLM, same code

Nothing above is SmolVLM-specific — `sink_scores` only needs an attention tensor plus the
image/text masks. Swapping the probe is the whole change. (Optional; needs a bigger GPU.)

In [ ]:
RUN_QWEN = False        # flip to True on a GPU with room for Qwen2.5-VL in 4-bit

if RUN_QWEN:
    Q = _load("probe_qwen", "tests/probe_qwen.py")
    oq = Q.make_qwen_output()
    if oq is None:
        print("qwen probe unavailable (auth / memory) - skipped")
    else:
        sq = VS.sink_scores(oq.post_softmax, oq.image_token_mask, oq.text_token_mask,
                            is_post_softmax=True)
        print(VS.sink_report(sq, VS.detect_sinks(sq)))
else:
    print("set RUN_QWEN = True to repeat the detection on Qwen2.5-VL")

## Notes

* **Do not ablate the sink inside the model.** It is load-bearing — our LOO run showed masking it
  causes the largest drop in answer logprob of any patch. Remove it from the *label*, the
  *candidate set* and the *selection budget*, but leave it in the forward pass. For FRM that means
  pinning it: `KEEP_g = sinks ∪ top_n(r over cand)`, so it always survives and never competes for
  the n slots.
* **`detect_sinks` adapts.** With no `k` it uses a median+MAD outlier rule, so it returns nothing
  when a model has no sink and several when it has several, capped at `max_frac` of the grid.
  Pass `k=` explicitly once you have looked at `sink_report` and decided.
* **Sweep the count.** `drop_sink_k=3` was never justified by evidence. Now you can vary the
  threshold and measure downstream precision@k instead of guessing.
* Estimating from ~6 varied examples is plenty; the per-example argmax printout above shows how
  stable it is.
* **Be honest about the quantile-vs-mean comparison.** On SmolVLM2 the naive mean also finds
  patch 80 — the sink there is dominant enough that either rule works. The quantile's advantage is
  demonstrated on synthetic data (where a content patch is built to beat the sink on mean incoming
  attention while losing on the floor), not yet on a real model. If the two columns agree on your
  model, that is information, not a failure: it means the sink is unambiguous.